# 4 — Embedding

Embeds chunked Q&A pairs using `intfloat/multilingual-e5-large`.

| | |
|---|---|
| **Input** | `../data/processed/chunks_simple.json` (Sprint 1) |
| **Future input** | `../data/processed/chunks_all.json` (after long chunker) |
| **Output** | `../data/processed/chunks_embedded.json` |
| **Failures** | `../data/processed/embed_failures.json` (only if errors occur) |

### Why multilingual-e5-large?
- Trained on 100+ languages including Turkish, which shares ~60% vocabulary with Azerbaijani
- Asymmetric retrieval design: documents use `passage:` prefix, queries use `query:` prefix
- 1024-dimensional vectors — better discrimination than mpnet's 768 dims
- Outperforms mpnet on MTEB multilingual retrieval benchmarks
- Runs fully locally — no API key, no data leaves your server

### E5 prefix rule (important)
At **index time** (here): prefix every chunk with `passage: `  
At **query time** (chatbot): prefix every user question with `query: `  
Skipping this degrades retrieval quality significantly.

## 0 — Install dependencies

In [ ]:
# Run once — comment out after first install
# !pip install sentence-transformers tqdm torch

## 1 — Imports & config

In [1]:
import json
import logging
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────

# INPUT: change to chunks_all.json once Sprint 2 (long chunker) is complete
INPUT_PATH   = Path("../data/processed/chunks_simple.json")
OUTPUT_PATH  = Path("../data/processed/chunks_embedded.json")
FAILURE_PATH = Path("../data/processed/embed_failures.json")

# Model — swap to multilingual-e5-base if RAM is tight (~1.1GB vs ~2.2GB)
MODEL_NAME = "intfloat/multilingual-e5-large"

# Batch size — 32 safe for CPU; increase to 64-128 on GPU
BATCH_SIZE = 32

# E5 document prefix — must be 'passage: ' for indexed documents
E5_DOC_PREFIX = "passage: "

# Normalize to unit length — required for cosine similarity
NORMALIZE = True

print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

Device: CUDA
Input : ..\data\processed\chunks_simple.json
Output: ..\data\processed\chunks_embedded.json


## 2 — Load & inspect chunks

In [3]:
with open(INPUT_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks loaded : {len(chunks):,}")
print(f"Chunk type breakdown: {dict(Counter(c['chunk_type'] for c in chunks))}")
print()

# Length stats
lengths = [len(c['chunk_text']) for c in chunks]
print(f"chunk_text length — min: {min(lengths)}, max: {max(lengths)}, mean: {sum(lengths)//len(lengths)} chars")

est_tokens = [l // 4 for l in lengths]
print(f"Est tokens         — min: {min(est_tokens)}, max: {max(est_tokens)}, mean: {sum(est_tokens)//len(est_tokens)}")
print(f"Est > 512 tokens   : {sum(1 for t in est_tokens if t > 512)} chunks")

Total chunks loaded : 3,558
Chunk type breakdown: {'simple': 3558}

chunk_text length — min: 119, max: 1514, mean: 885 chars
Est tokens         — min: 29, max: 378, mean: 220
Est > 512 tokens   : 0 chunks


In [4]:
# Preview a chunk — see what the model will actually receive
sample = chunks[0]
print("── Chunk fields ──────────────────────────────────────")
for k, v in sample.items():
    print(f"  {k:<16}: {str(v)[:100]}")
print()
print("── Prefixed text (what model receives) ──────────────")
prefixed = E5_DOC_PREFIX + sample['chunk_text']
print(prefixed[:300], "..." if len(prefixed) > 300 else "")

── Chunk fields ──────────────────────────────────────
  chunk_text      : Sual: Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətən
  question        : Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətəndaşı o
  answer          : Bildiririk ki, hüquqi şəxsin ləğv edilməsi proseduru, o cümlədən təqdim olunmalı sənədlərin siyahısı
  source_id       : 10826
  source_url      : https://www.taxes.gov.az/az/page/suallar-ve-cavablar
  chunk_index     : 0
  total_chunks    : 1
  chunk_type      : simple
  answer_date     : 2026-04-23
  read_count      : 62
  average_rate    : 0
  rate_count      : 0
  question_len    : 374
  answer_len      : 278
  combined_len    : 652
  chunked_at      : 2026-05-11T11:15:11.949612+00:00

── Prefixed text (what model receives) ──────────────
passage: Sual: Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətəndaşı olan bir nəfərdir. H

## 3 — Load model

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Loading {MODEL_NAME} on {device} ...")

t0 = time.time()
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Embedding dim  : {model.get_embedding_dimension()}")
print(f"Max seq length : {model.max_seq_length} tokens")

16:13:24 | INFO | Loading intfloat/multilingual-e5-large on cuda ...
16:13:25 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
16:13:25 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
16:13:25 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:13:25 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-large.
16:13:25 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:13:25 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
16:13:25 | INFO | HTTP Reques

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

16:13:27 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
16:13:27 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:13:27 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
16:13:28 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:13:28 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
16:13:28 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/tokenizer_config.json "HTTP/1.1 200 OK"
16:13:28 | INFO | HTTP R

Model loaded in 13.2s
Embedding dim  : 1024
Max seq length : 512 tokens


## 4 — Sanity check: embed one sample

In [6]:
# Quick single-vector test before committing to the full run
test_doc = E5_DOC_PREFIX + chunks[0]['chunk_text']
test_vec = model.encode([test_doc], normalize_embeddings=NORMALIZE, convert_to_numpy=True)

print(f"Vector shape : {test_vec.shape}")
print(f"Vector norm  : {round(float((test_vec**2).sum()**0.5), 6)}  (should be ~1.0 if normalized)")
print(f"First 5 dims : {[round(float(v), 6) for v in test_vec[0][:5]]}")
print()
print("Sanity check passed — model is ready for full run")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vector shape : (1, 1024)
Vector norm  : 1.0  (should be ~1.0 if normalized)
First 5 dims : [0.006775, -0.00686, -0.01399, -0.035928, 0.023459]

Sanity check passed — model is ready for full run


## 5 — Embed all chunks

In [11]:
pip install --upgrade jupyter notebook ipywidgets tqdm

  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached argon2_cffi_bindings-25.1.0-cp39-abi3-win_amd64.whl.metadata (7.5 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl.metadata (4.2 kB)
  Using cached rfc3339_validator-0.1.4-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached rfc3986_validator-0.1.1-py2.py3-none-any.whl.met

In [15]:
pip install --upgrade jupyterlab ipywidgets widgetsnbextension


Note: you may need to restart the kernel to use updated packages.


In [17]:
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
Building jupyterlab assets (production, minimized)


In [12]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook qtconsole run script server
troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [7]:
# Prepare all texts with E5 document prefix
texts = [E5_DOC_PREFIX + c["chunk_text"] for c in chunks]

# Split into batches
batches = [texts[i : i + BATCH_SIZE] for i in range(0, len(texts), BATCH_SIZE)]
print(f"Embedding {len(texts):,} chunks in {len(batches):,} batches ...")

vectors = [None] * len(texts)
failed_indices = []
t_start = time.time()

for batch_num, batch_texts in enumerate(tqdm(batches, desc="Embedding batches", unit="batch")):
    start_idx = batch_num * BATCH_SIZE
    try:
        vecs = model.encode(
            batch_texts,
            normalize_embeddings=NORMALIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
        for i, vec in enumerate(vecs):
            vectors[start_idx + i] = vec.tolist()
    except Exception as e:
        log.warning(f"Batch {batch_num} failed: {e}")
        for i in range(len(batch_texts)):
            failed_indices.append(start_idx + i)

elapsed = time.time() - t_start
succeeded = sum(1 for v in vectors if v is not None)
print(f"\nDone in {elapsed:.1f}s — {succeeded:,} succeeded, {len(failed_indices):,} failed")

Embedding 3,558 chunks in 112 batches ...


Embedding batches:   0%|          | 0/112 [00:00<?, ?batch/s]


Done in 313.2s — 3,558 succeeded, 0 failed


## 6 — Attach embeddings and metadata to chunks

In [8]:
embedding_dim = model.get_sentence_embedding_dimension()
failed_set    = set(failed_indices)
ts            = datetime.now(timezone.utc).isoformat()

embedded = []
failed   = []

for idx, (chunk, vec) in enumerate(zip(chunks, vectors)):
    if idx in failed_set or vec is None:
        failed.append({
            "source_id":   chunk["source_id"],
            "chunk_index": chunk["chunk_index"],
            "chunk_type":  chunk["chunk_type"],
            "reason":      "embedding_failed",
        })
        continue

    embedded.append({
        **chunk,                           # all original fields preserved
        "embedding":       vec,            # the vector
        "embedding_model": MODEL_NAME,     # traceability
        "embedding_dim":   embedding_dim,  # for DB schema validation
        "e5_prefix_used":  E5_DOC_PREFIX,  # reminder: use 'query: ' at search time
        "normalized":      NORMALIZE,
        "embedded_at":     ts,
    })

print(f"Embedded chunks : {len(embedded):,}")
print(f"Failed chunks   : {len(failed):,}")

Embedded chunks : 3,558
Failed chunks   : 0


C:\Users\elihu\AppData\Local\Temp\ipykernel_29524\4134148433.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


## 7 — Spot-check an embedded chunk

In [9]:
if embedded:
    s = embedded[0]
    print("── Fields on embedded chunk ──────────────────────────")
    for k, v in s.items():
        if k == "embedding":
            print(f"  {'embedding':<16}: [{', '.join(str(round(x,6)) for x in v[:5])}, ...] ({len(v)} dims)")
        else:
            print(f"  {k:<16}: {str(v)[:100]}")

    # Verify normalization
    norm = round(sum(x**2 for x in s['embedding'])**0.5, 6)
    print(f"\nVector norm: {norm}  ({'✅ normalized' if abs(norm - 1.0) < 0.001 else '⚠ not normalized'})")

── Fields on embedded chunk ──────────────────────────
  chunk_text      : Sual: Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətən
  question        : Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətəndaşı o
  answer          : Bildiririk ki, hüquqi şəxsin ləğv edilməsi proseduru, o cümlədən təqdim olunmalı sənədlərin siyahısı
  source_id       : 10826
  source_url      : https://www.taxes.gov.az/az/page/suallar-ve-cavablar
  chunk_index     : 0
  total_chunks    : 1
  chunk_type      : simple
  answer_date     : 2026-04-23
  read_count      : 62
  average_rate    : 0
  rate_count      : 0
  question_len    : 374
  answer_len      : 278
  combined_len    : 652
  chunked_at      : 2026-05-11T11:15:11.949612+00:00
  embedding       : [0.006775, -0.00686, -0.01399, -0.035928, 0.023459, ...] (1024 dims)
  embedding_model : intfloat/multilingual-e5-large
  embedding_dim   : 1024
  e5_prefix_used  : pass

## 8 — Quick similarity test

In [10]:
# Test: embed a sample query and find closest chunks
# This validates that retrieval will work before touching the DB
import numpy as np

test_query = "query: VÖEN-in ləğvi proseduru necədir?"
query_vec  = model.encode([test_query], normalize_embeddings=True, convert_to_numpy=True)[0]

# Cosine similarity = dot product on normalized vectors
all_vecs = np.array([c["embedding"] for c in embedded])
scores   = all_vecs @ query_vec
top5_idx = scores.argsort()[::-1][:5]

print(f"Query: {test_query}")
print()
for rank, idx in enumerate(top5_idx, 1):
    chunk = embedded[idx]
    print(f"Rank {rank} | score: {scores[idx]:.4f} | source_id: {chunk['source_id']}")
    print(f"  Q: {chunk['question'][:120]}")
    print(f"  A: {chunk['answer'][:120]}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: query: VÖEN-in ləğvi proseduru necədir?

Rank 1 | score: 0.8722 | source_id: 9298
  Q: Salam. VÖEN-mi necə ləğv edə bilərəm?
  A: Bildiririk ki, vergi uçotundan çıxmaq üçün vergi ödəyicisi və ya onun səlahiyyətli nümayəndəsi (notarial qaydada təsdiq 

Rank 2 | score: 0.8684 | source_id: 1516
  Q: VÖEN-in ləğv edilməsi.
  A: Bildiririk ki, vergi uçotunuzun ləğv edilməsi üçün “Fiziki şəxsin vergi uçotundan çıxarılması haqqında ərizə” və uçota a

Rank 3 | score: 0.8680 | source_id: 4651
  Q: 2004237452 nömrəli VÖEN-in ləğv olunması ücün uzun muddətdir ki, muraciət etmişəm. Ancaq hələ ləğv olunmayıb, xahiş olun
  A: Bildiririk ki, fiziki şəxs vergi uçotunun ləğv edilməsi haqda müraciət etdikdə, vergi orqanında müvafiq prosedurlar (akt

Rank 4 | score: 0.8679 | source_id: 238
  Q: Salam, mən 2014-cü ildə VÖEN açdırmışam, heç bir sahibkarlıqla məşğul  olmamışam və ehtiyac olmadığı üçün VÖEN-i necə lə
  A: Bildiririk ki, vergi uçotunuzun ləğv edilməsi üçün “Fiziki şəxsin vergi uçotunda

## 9 — Save outputs

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save embedded chunks (no indent — keeps file size manageable)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(embedded, f, ensure_ascii=False)
print(f"Saved {len(embedded):,} embedded chunks → {OUTPUT_PATH}")

# Save failures only if there are any
if failed:
    with open(FAILURE_PATH, "w", encoding="utf-8") as f:
        json.dump(failed, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(failed):,} failures → {FAILURE_PATH}")
else:
    print("No failures — embed_failures.json not created")

Saved 3,558 embedded chunks → ..\data\processed\chunks_embedded.json
No failures — embed_failures.json not created


## 10 — Final summary

In [13]:
print("═" * 52)
print("  EMBEDDING COMPLETE")
print("═" * 52)
print(f"  Input chunks      : {len(chunks):,}")
print(f"  Embedded          : {len(embedded):,}")
print(f"  Failed            : {len(failed):,}")
print(f"  Model             : {MODEL_NAME}")
print(f"  Embedding dim     : {embedding_dim}")
print(f"  Normalized        : {NORMALIZE}")
print(f"  Time elapsed      : {elapsed:.1f}s")
print(f"  Output            : {OUTPUT_PATH}")
print("═" * 52)


════════════════════════════════════════════════════
  EMBEDDING COMPLETE
════════════════════════════════════════════════════
  Input chunks      : 3,558
  Embedded          : 3,558
  Failed            : 0
  Model             : intfloat/multilingual-e5-large
  Embedding dim     : 1024
  Normalized        : True
  Time elapsed      : 313.2s
  Output            : ..\data\processed\chunks_embedded.json
════════════════════════════════════════════════════
